# LLM Fine-tuning + Eval — Resume → JSON (Colab / Kaggle T4)

Runs the full project on a **free T4**: install → build data → QLoRA fine-tune → base-vs-fine-tuned eval → view the comparison table.

**Before running:** set `Runtime → Change runtime type → GPU (T4)`.

The base model (`Qwen/Qwen2.5-3B-Instruct`) is open — no HF login or API key needed.

## 1. Get the code
Clone your repo, or upload the project folder. Edit `GITHUB_URL` to your fork.

In [ ]:
import os

GITHUB_URL = "https://github.com/<your-username>/llm-finetune-eval.git"  # <-- edit me
REPO_DIR = "/content/llm-finetune-eval"

if not os.path.isdir(REPO_DIR):
    !git clone $GITHUB_URL $REPO_DIR
%cd $REPO_DIR
!ls

## 2. Install dependencies
`bitsandbytes` is the 4-bit quantization backend (CUDA only).

In [ ]:
!pip install -q -r requirements.txt
import torch; print('CUDA available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no GPU')

## 3. Build the dataset (deterministic)
The splits are already committed; this just confirms they regenerate identically. The **test split is frozen**.

In [ ]:
!python data/build_dataset.py --n 400 --seed 42

## 4. QLoRA fine-tune
4-bit base + LoRA adapter. ~10–20 min on a T4 for 2 epochs. Watch the train/eval loss gap for overfitting.
Saves the adapter to `adapters/qwen-resume/` and a loss curve to `assets/loss_curve.png`.

In [ ]:
!python train.py --base Qwen/Qwen2.5-3B-Instruct --epochs 2

In [ ]:
from IPython.display import Image
Image('assets/loss_curve.png')

## 5. Evaluate base vs. fine-tuned
Generates predictions for BOTH models on the same frozen test set, scores them, and writes `results/comparison.md` + charts.
Raw predictions are saved to `results/raw/` so you can re-score offline later without a GPU.

In [ ]:
!python eval/run_eval.py --base Qwen/Qwen2.5-3B-Instruct --adapter adapters/qwen-resume

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('results/comparison.md').read()))
display(Image('results/charts/headline.png'))
display(Image('results/charts/per_field_f1.png'))

## 6. (Optional) Merge + push to the Hugging Face Hub
Run `huggingface-cli login` first, then merge the adapter into fp16 weights and push with a model card.

In [ ]:
# from huggingface_hub import notebook_login; notebook_login()
# !python merge_and_push.py --adapter adapters/qwen-resume --push <your-username>/qwen2.5-3b-resume-json

## 7. Download artifacts
Pull the results back to your machine to commit them to the repo (the comparison table + charts are your strongest interview artifact).

In [ ]:
import shutil
shutil.make_archive('/content/artifacts', 'zip', '.', 'results')
shutil.make_archive('/content/adapter', 'zip', 'adapters', 'qwen-resume')
try:
    from google.colab import files
    files.download('/content/artifacts.zip')
    files.download('/content/adapter.zip')
except Exception as e:
    print('not on Colab or download unavailable:', e)